# Basic Local Aignment Search Tool

This notebook gives some quick info to the BLASTing in the project. All the direct work on BLAST takes place outside of this project folder, as it is problematic connecting OneDrive with BLAST. In my experience. The work was done with Git Bash.\
The reference genome was downloaded as such:

Further on, from this, the database was created, as one needs to have something to BLAST against. The database is included in ecoli_db. Just for show...\
Then, the BLAST could proceed. The file obtained for transporters is named approac1.fasta. The limits can also easily be adjusted. Here, it is shown with E-value < 1e-5. -outfmt is the tabular output.

The output looks like this:

Where the columns respectively are:\
Query_ID  Subject_ID  Identity  Alignment_Length  Mismatches  Gap_Openings  Query_Start  Query_End  Subject_Start  Subject_End  E-value  Bit-Score

Next up, the results were obtained, and could also be filtered to only retain the highest scoring sequences. This filter keeps only the instances where the Identity > 40% matched, and E-value < 1e-5. The E-val part is in this example useless, as it was already applied in the BLASTing section, but can easily be tuned.

Respectively, all the matches can easily be counted, and all the IDs from approach1.fasta can also be listed as such:

The file is ready for use and further processing now. Well, it was before these last lines as well. But they just give a nice and quick overview.

Below follows an example of how the file can be processed to obtain the desired info on the transport reactions of E. coli.

In [3]:
import pandas as pd

In [ ]:
blast_results = pd.read_csv("results_a1.txt", sep="\t", header=None)

blast_results.columns = [
    "QueryID", "SubjectID", "Identity", "AlignLength", "Mismatches", 
    "GapOpens", "QStart", "QEnd", "SStart", "SEnd", "E-value", "BitScore"
]

eval_threshold = 1e-5
id_threshold = 30

filtered_hits = blast_results[
    (blast_results["E-value"] <= eval_threshold) & 
    (blast_results["Identity"] >= id_threshold)
]

matched_ids = set(filtered_hits["QueryID"].tolist())
matched_ids
filtered_hits[["UID", "TCID"]] = filtered_hits["QueryID"].str.split("|", expand=True)
filtered_hits = filtered_hits.drop(columns=['QueryID'])
cols = ['UID', 'TCID'] + [col for col in filtered_hits.columns if col not in ['UID', 'TCID']]
filtered_hits = filtered_hits[cols]

Now, merging based on the data BLASTed againts.

In [ ]:
a1 = pd.read_csv("../Approach 1/a1_df.tsv", sep="\t")
results = pd.merge(a1, filtered_hits, on=["UID", "TCID"], how="inner")

Alright, this concludes the BLAST-section. The columns and what to keep, remain for the pipeline to take control of. It is better to keep as much info as possible until the final pipeline is done. Then, optimization might be correct to put in place. Above is described a way to solve the issue on merging BLAST results against the transporters from TCDB. Next step is to actually build the pipeline!